# 🐍 Day 7: Mini-Project - Blog Application

- Full CRUD blog with Flask
- SQLAlchemy models (Post, User)
- Templates with inheritance
- Create, read, update, delete posts

## Project Overview

We'll build a blog with:
- **Post** model: id, title, content, created_at
- **User** model: id, username (simplified)
- List posts, view single post, create, edit, delete
- Jinja2 templates with base layout

## Step 1: Models

Define Post and User with SQLAlchemy.

In [ ]:
from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from datetime import datetime

app = Flask(__name__)
app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///blog.db"
app.config["SECRET_KEY"] = "dev-secret"
db = SQLAlchemy(app)

class User(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(80), unique=True, nullable=False)

class Post(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    title = db.Column(db.String(200), nullable=False)
    content = db.Column(db.Text, nullable=False)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)
    user_id = db.Column(db.Integer, db.ForeignKey("user.id"), nullable=True)
    user = db.relationship("User", backref="posts")

with app.app_context():
    db.create_all()
print("Models and DB created")

## Step 2: Base Template

Create templates/base.html with nav and content block.

In [ ]:
import os
os.makedirs("templates", exist_ok=True)

base = """<!DOCTYPE html>
<html>
<head><title>{% block title %}Blog{% endblock %}</title></head>
<body>
  <nav><a href="/">Home</a> | <a href="/posts/new">New Post</a></nav>
  <hr>
  {% block content %}{% endblock %}
</body>
</html>"""

with open("templates/base.html", "w") as f:
    f.write(base)
print("Created base.html")

## Step 3: List and View Templates

Create index and post detail templates.

In [ ]:
index_tpl = """{% extends "base.html" %}
{% block title %}Posts{% endblock %}
{% block content %}
  <h1>Posts</h1>
  {% for post in posts %}
    <h2><a href="/posts/{{ post.id }}">{{ post.title }}</a></h2>
    <p>{{ post.created_at.strftime('%Y-%m-%d') }}</p>
  {% else %}
    <p>No posts yet.</p>
  {% endfor %}
{% endblock %}"""

post_tpl = """{% extends "base.html" %}
{% block title %}{{ post.title }}{% endblock %}
{% block content %}
  <h1>{{ post.title }}</h1>
  <p>{{ post.created_at.strftime('%Y-%m-%d %H:%M') }}</p>
  <pre>{{ post.content }}</pre>
  <a href="/posts/{{ post.id }}/edit">Edit</a>
  <form action="/posts/{{ post.id }}/delete" method="POST" style="display:inline">
    <button type="submit">Delete</button>
  </form>
{% endblock %}"""

with open("templates/index.html", "w") as f:
    f.write(index_tpl)
with open("templates/post.html", "w") as f:
    f.write(post_tpl)
print("Created index.html, post.html")

## Step 4: Form Template

Create form for new/edit post.

In [ ]:
form_tpl = """{% extends "base.html" %}
{% block title %}{{ title }}{% endblock %}
{% block content %}
  <h1>{{ title }}</h1>
  <form method="POST">
    <p><label>Title: <input name="title" value="{{ post.title if post else '' }}" size="50"></label></p>
    <p><label>Content:<br><textarea name="content" rows="10" cols="50">{{ post.content if post else '' }}</textarea></label></p>
    <button type="submit">Save</button>
  </form>
{% endblock %}"""

with open("templates/post_form.html", "w") as f:
    f.write(form_tpl)
print("Created post_form.html")

## Step 5: Routes - List and View

GET / and GET /posts/<id>

In [ ]:
from flask import render_template, request, redirect, url_for

@app.route("/")
def index():
    posts = Post.query.order_by(Post.created_at.desc()).all()
    return render_template("index.html", posts=posts)

@app.route("/posts/<int:id>")
def view_post(id):
    post = Post.query.get_or_404(id)
    return render_template("post.html", post=post)

print("Routes: index, view_post")

## Step 6: Routes - Create

GET /posts/new (form) and POST /posts/new (create)

In [ ]:
@app.route("/posts/new", methods=["GET", "POST"])
def new_post():
    if request.method == "POST":
        title = request.form.get("title", "").strip()
        content = request.form.get("content", "").strip()
        if title and content:
            post = Post(title=title, content=content)
            db.session.add(post)
            db.session.commit()
            return redirect(url_for("view_post", id=post.id))
    return render_template("post_form.html", title="New Post", post=None)

print("Route: new_post")

## Step 7: Routes - Edit and Delete

GET/POST /posts/<id>/edit and POST /posts/<id>/delete

In [ ]:
@app.route("/posts/<int:id>/edit", methods=["GET", "POST"])
def edit_post(id):
    post = Post.query.get_or_404(id)
    if request.method == "POST":
        post.title = request.form.get("title", "").strip()
        post.content = request.form.get("content", "").strip()
        db.session.commit()
        return redirect(url_for("view_post", id=post.id))
    return render_template("post_form.html", title="Edit Post", post=post)

@app.route("/posts/<int:id>/delete", methods=["POST"])
def delete_post(id):
    post = Post.query.get_or_404(id)
    db.session.delete(post)
    db.session.commit()
    return redirect(url_for("index"))

print("Routes: edit_post, delete_post")

## Step 8: Seed Data and Run

Add sample post and run the app.

In [ ]:
with app.app_context():
    if Post.query.count() == 0:
        p = Post(title="Welcome", content="This is the first post. Edit or delete it!")
        db.session.add(p)
        db.session.commit()
        print("Added sample post")
    else:
        print("Posts exist:", Post.query.count())

# Run: app.run(port=5000)
print("Run: app.run(port=5000)")

## Complete Application Summary

- **Models**: Post, User with SQLAlchemy
- **Templates**: base, index, post, post_form
- **Routes**: index, view_post, new_post, edit_post, delete_post
- **CRUD**: Create (new_post), Read (index, view_post), Update (edit_post), Delete (delete_post)

## Extensions to Try

- Add Flask-Login and require auth for create/edit/delete
- Add pagination to index
- Add markdown support for content
- Add comments model and display
- Add search by title

## Key Takeaways

- Flask-SQLAlchemy for models and db session
- Template inheritance for consistent layout
- CRUD: form GET/POST, redirect after success
- get_or_404 for missing resources

## 🎉 Week 14 Complete!

You've built a full Flask blog with templates, forms, SQLAlchemy, auth, and REST APIs. Next week: FastAPI!